In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('e-commerce_delivery_data.csv')

In [3]:
df['courier_partner'] = df['courier_partner'].str.strip().str.lower()
df['courier_partner'].unique()
df['courier_partner'] = df['courier_partner'].replace(['sa paribahan','s.a. paribahan'],'sa paribahan')
df['courier_partner'] = df['courier_partner'].replace(['ecourier','e-courier'], 'e-courier')
df['courier_partner'] = df['courier_partner'].replace(['redx','red x'],'redx')
df['courier_partner'] = df['courier_partner'].replace(['sundarban','sundorbon'],'sundarban')
df['courier_partner'].unique()

df['order_value'] = df['order_value'].fillna(df['order_value'].mean())
df['distance_km'] = df['distance_km'].fillna(df['distance_km'].mean())
df['customer_rating_history'] = df['customer_rating_history'].fillna(df['customer_rating_history'].mean())
df = df.dropna(subset=['courier_partner'])


In [4]:
df['category'] = df['category'].str.strip().str.lower()

In [5]:
df['category'].unique()

<StringArray>
[        'clothing',          'grocery',      'electronics',
   'home & kitchen',        'groceries',           'beauty',
 'home and kitchen',            'books']
Length: 8, dtype: str

In [6]:
df['category'] = df['category'].replace(['grocery','groceries'],'groceries')
df['category'] = df['category'].replace(['home & kitchen', 'home and kitchen'], 'home & kitchen')
df['category'].unique()



<StringArray>
['clothing', 'groceries', 'electronics', 'home & kitchen', 'beauty', 'books']
Length: 6, dtype: str

In [7]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['order_month'] = df['order_date'].dt.month
df['order_of_the_week'] = df['order_date'].dt.dayofweek

In [8]:
df = pd.get_dummies(df, columns=['category','warehouse_city','courier_partner','payment_method'])

In [9]:
df.shape

(8935, 29)

In [10]:
df.head()

,order_id,order_date,order_value,distance_km,customer_rating_history,num_items,promo_applied,delivery_status,order_month,order_of_the_week,...,warehouse_city_Rajshahi,warehouse_city_Sylhet,courier_partner_e-courier,courier_partner_pathao,courier_partner_redx,courier_partner_sa paribahan,courier_partner_sundarban,payment_method_Card,payment_method_Cash on Delivery,payment_method_Mobile Banking
0,103863,2023-11-18,286.430000,21.9,4.7,4,0,Delayed,11,5,...,False,False,False,False,False,True,False,False,True,False
1,101006,2024-02-01,150.000000,77.4,4.4,5,1,Delayed,2,3,...,False,False,True,False,False,False,False,False,False,True
2,101271,2023-09-16,1809.639964,1.0,4.0,1,0,On-Time,9,5,...,False,True,False,False,False,False,True,False,True,False
3,107757,2024-06-08,2200.840000,29.2,2.8,2,0,Delayed,6,5,...,False,False,False,False,True,False,False,True,False,False
4,107076,2024-05-27,2868.460000,7.7,5.0,1,0,On-Time,5,0,...,False,False,False,False,False,False,True,True,False,False


In [11]:
from sklearn.model_selection import train_test_split

In [12]:
y = df['delivery_status']
x = df.drop(columns=['delivery_status','order_id','order_date'])
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [13]:
x_train.shape


(7148, 26)

In [14]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train_scaled = sc.fit_transform(x_train)
x_test_scaled = sc.transform(x_test)


In [15]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(x_train_scaled, y_train)
y_pred = model.predict(x_test_scaled)

In [16]:
print('Y prediction is',y_pred[:10])
print('y test is ', y_test[:10].values)

Y prediction is ['On-Time' 'On-Time' 'On-Time' 'On-Time' 'On-Time' 'On-Time' 'On-Time'
 'On-Time' 'On-Time' 'Delayed']
y test is  <StringArray>
['On-Time', 'Delayed', 'On-Time', 'Delayed', 'Delayed', 'On-Time', 'Delayed',
 'On-Time', 'Delayed', 'Delayed']
Length: 10, dtype: str


In [17]:
df['delivery_status'].unique()

<StringArray>
['Delayed', 'On-Time', 'Cancelled']
Length: 3, dtype: str

In [18]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred, labels=['Delayed', 'On-Time', 'Cancelled'])
print(cm)

[[  70  486    0]
 [  61 1107    0]
 [  16   47    0]]


In [19]:
y_pred = model.predict(x_test_scaled)
print(y_pred[:10])
cm = confusion_matrix(y_test, y_pred, labels=['On-Time', 'Delayed', 'Cancelled'])
print(cm)

['On-Time' 'On-Time' 'On-Time' 'On-Time' 'On-Time' 'On-Time' 'On-Time'
 'On-Time' 'On-Time' 'Delayed']
[[1107   61    0]
 [ 486   70    0]
 [  47   16    0]]


In [20]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred,))

              precision    recall  f1-score   support

   Cancelled       0.00      0.00      0.00        63
     Delayed       0.48      0.13      0.20       556
     On-Time       0.68      0.95      0.79      1168

    accuracy                           0.66      1787
   macro avg       0.38      0.36      0.33      1787
weighted avg       0.59      0.66      0.58      1787



C:\Users\masud\PycharmProject\e-commerce_delivery\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\masud\PycharmProject\e-commerce_delivery\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\masud\PycharmProject\e-commerce_delivery\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

In [26]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

dt = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced')
dt.fit(x_train_scaled, y_train)
dt_pred = dt.predict(x_test_scaled)

rf = RandomForestClassifier(max_depth=5, random_state=42, n_estimators=100, class_weight='balanced')
rf.fit(x_train_scaled, y_train)
rf_pred = rf.predict(x_test_scaled)


In [27]:
print('Random forest',classification_report(y_test,rf_pred))

Random forest               precision    recall  f1-score   support

   Cancelled       0.07      0.35      0.11        63
     Delayed       0.37      0.19      0.25       556
     On-Time       0.71      0.71      0.71      1168

    accuracy                           0.54      1787
   macro avg       0.38      0.42      0.36      1787
weighted avg       0.58      0.54      0.55      1787



In [28]:
print('decision tree',classification_report(y_test,dt_pred))

decision tree               precision    recall  f1-score   support

   Cancelled       0.06      0.32      0.10        63
     Delayed       0.37      0.18      0.24       556
     On-Time       0.70      0.71      0.70      1168

    accuracy                           0.53      1787
   macro avg       0.37      0.40      0.35      1787
weighted avg       0.57      0.53      0.54      1787



In [25]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(x_train_scaled, y_train)
y_pred = model.predict(x_test_scaled)

from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred,))

              precision    recall  f1-score   support

   Cancelled       0.05      0.33      0.09        63
     Delayed       0.40      0.29      0.34       556
     On-Time       0.73      0.62      0.67      1168

    accuracy                           0.51      1787
   macro avg       0.40      0.42      0.37      1787
weighted avg       0.61      0.51      0.55      1787



In [29]:
importance = pd.Series(rf.feature_importances_, index=x.columns).sort_values(ascending=False)
print(importance.head(10))

distance_km                        0.348237
customer_rating_history            0.086477
order_value                        0.084370
courier_partner_sundarban          0.080592
payment_method_Cash on Delivery    0.060321
order_month                        0.048736
order_of_the_week                  0.040443
num_items                          0.037246
payment_method_Mobile Banking      0.024821
category_books                     0.017547
dtype: float64


In [31]:
df.groupby('delivery_status')[['payment_method_Cash on Delivery', 'payment_method_Card', 'payment_method_Mobile Banking']].mean()

,payment_method_Cash on Delivery,payment_method_Card,payment_method_Mobile Banking
delivery_status,,,
Cancelled,0.675497,0.155629,0.168874
Delayed,0.608871,0.176686,0.214443
On-Time,0.529721,0.206097,0.264183


In [35]:
import joblib

joblib.dump(rf, 'delivery_model.pkl')
joblib.dump(sc, 'delivery_scaler.pkl')

['delivery_scaler.pkl']

In [1]:
import joblib

import pandas as pd
rf_model_loaded = joblib.load('delivery_model.pkl')
sc_model_loaded = joblib.load('delivery_scaler.pkl')

In [2]:
def predict_delivery_status(order_details, training_column):
    new_order_df = pd.DataFrame([order_details])
    new_order_encode = pd.get_dummies(new_order_df)
    new_order_alligned = new_order_encode.reindex(columns=training_column, fill_value=0)
    new_order_scaled = sc_model_loaded.transform(new_order_alligned)
    predict = rf_model_loaded.predict(new_order_scaled)
    return predict[0]

In [3]:
training_columns = ['order_value', 'distance_km', 'customer_rating_history', 'num_items',
    'promo_applied', 'order_month', 'order_of_the_week',
    'category_beauty', 'category_books', 'category_clothing', 'category_electronics',
    'category_groceries', 'category_home & kitchen',
    'warehouse_city_Chattogram', 'warehouse_city_Dhaka', 'warehouse_city_Khulna',
    'warehouse_city_Rajshahi', 'warehouse_city_Sylhet',
    'courier_partner_e-courier', 'courier_partner_pathao', 'courier_partner_redx',
    'courier_partner_sa paribahan', 'courier_partner_sundarban',
    'payment_method_Card', 'payment_method_Cash on Delivery', 'payment_method_Mobile Banking']

joblib.dump(training_columns, 'delivery_columns.pkl')

['delivery_columns.pkl']

In [4]:
new_order = {
    'order_value': 3200,
    'distance_km': 45,
    'customer_rating_history': 4.2,
    'num_items': 3,
    'promo_applied': 1,
    'order_month': 7,
    'order_of_the_week': 5,
    'category_electronics': 1,
    'warehouse_city_Dhaka': 1,
    'courier_partner_sundarban': 1,
    'payment_method_Cash on Delivery': 1
}

result = predict_delivery_status(new_order, training_columns)
print(result)

Cancelled


In [5]:
import streamlit as st
import pandas as pd
import joblib

In [6]:
rf_model_loaded = joblib.load('delivery_model.pkl')
sc_model_loaded = joblib.load('delivery_scaler.pkl')
training_columns = joblib.load('delivery_columns.pkl')

In [7]:
st.title('Delivery outcome predictor')
st.write("Enter a  new order details to find out if the order get delayed, on time, or cancelled.")

2026-09-08 15:20:41.146 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:20:41.867 
  command:

    streamlit run C:\Users\masud\PycharmProject\e-commerce_delivery\.venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-09-08 15:20:41.868 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:20:41.870 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:20:41.872 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:20:41.873 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:20:41.875 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [8]:
order_value = st.number_input("Order Value (taka)", min_value=100, max_value=30000, value=2000)
distance_km = st.number_input("Distance (km)", min_value=1, max_value=150, value=15)
customer_rating_history = st.slider("Customer Rating History", 1.0, 5.0, 4.0)
num_items = st.number_input("Number of Items", min_value=1, max_value=15, value=2)
promo_applied = st.selectbox("Promo Applied?", ["No", "Yes"])
order_month = st.selectbox("Order Month", list(range(1, 13)))
order_dayofweek = st.selectbox("Day of Week (0=Mon, 6=Sun)", list(range(0, 7)))

category = st.selectbox("Category", ["electronics", "clothing", "groceries", "home & kitchen", "beauty", "books"])
warehouse_city = st.selectbox("Warehouse City", ["Dhaka", "Chattogram", "Sylhet", "Khulna", "Rajshahi"])
courier_partner = st.selectbox("Courier Partner", ["pathao", "redx", "e-courier", "sa paribahan", "sundarban"])
payment_method = st.selectbox("Payment Method", ["Cash on Delivery", "Card", "Mobile Banking"])

# --- When the button is clicked, build the row and predict ---
if st.button("Predict Delivery Outcome"):

    # Step 1: build the raw order as a dictionary
    order_details = {
        'order_value': order_value,
        'distance_km': distance_km,
        'customer_rating_history': customer_rating_history,
        'num_items': num_items,
        'promo_applied': 1 if promo_applied == "Yes" else 0,
        'order_month': order_month,
        'order_of_the_week': order_dayofweek,
        f'category_{category}': 1,
        f'warehouse_city_{warehouse_city}': 1,
        f'courier_partner_{courier_partner}': 1,
        f'payment_method_{payment_method}': 1,
    }

    # Step 2: turn it into a dataframe, one-hot encode, align columns (same as before)
    new_order_df = pd.DataFrame([order_details])
    new_order_encoded = pd.get_dummies(new_order_df)
    new_order_aligned = new_order_encoded.reindex(columns=training_columns, fill_value=0)

    # Step 3: scale using the SAME scaler from training
    new_order_scaled = sc_model_loaded.transform(new_order_aligned)

    # Step 4: predict
    prediction = rf_model_loaded.predict(new_order_scaled)[0]
    probabilities = rf_model_loaded.predict_proba(new_order_scaled)[0]

    st.subheader(f"Prediction: {prediction}")

    # Show confidence for each class
    st.write("Confidence breakdown:")
    prob_df = pd.DataFrame({
        'Outcome': rf_model_loaded.classes_,
        'Probability': probabilities
    }).sort_values('Probability', ascending=False)
    st.dataframe(prob_df)

2026-09-08 15:21:23.057 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:21:23.060 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:21:23.061 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:21:23.063 Session state does not function when running a script without `streamlit run`
2026-09-08 15:21:23.064 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:21:23.065 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:21:23.066 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:21:23.067 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:21